## 0. Install & Import

In [5]:
!pip install --upgrade groq -q

In [ ]:
import os, json, time, re
from typing import List, Dict, Any, Optional
from collections import defaultdict
from groq import Groq

# ── CONFIGURATION ──────────────────────────────────────────────────
GROQ_API_KEY = os.getenv('GROQ_API_KEY')  # No fallback!
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY environment variable not set")

# Fallback chain — all free on Groq
MODEL_CHAIN          = ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']

SIMILARITY_THRESHOLD = 0.80
MAX_ITERATIONS       = 5
BATCH_SIZE           = 3      # small → stays under 30 RPM / 6000 TPM
INTER_BATCH_SLEEP    = 8      # seconds between batches

GROUND_TRUTH_PATH  = 'summary.json'
FORMAL_SPECS_PATH  = 'formal_specifications.json'
DRAFT_SUMMARY_PATH = 'draft_summary.json'
JUDGE_REPORT_PATH  = 'judge_report.json'
ITER_LOG_PATH      = 'iteration_log.json'

# ── INIT ───────────────────────────────────────────────────────────
client           = Groq(api_key=GROQ_API_KEY)
active_model_idx = 0

def current_model() -> str:
    return MODEL_CHAIN[active_model_idx]

print(f'Provider : Groq (free, rate-limited only — no daily quota)')
print(f'Models   : {MODEL_CHAIN}')
print(f'Threshold: {SIMILARITY_THRESHOLD}  Batch: {BATCH_SIZE}  Sleep: {INTER_BATCH_SLEEP}s')

Provider : Groq (free, rate-limited only — no daily quota)
Models   : ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']
Threshold: 0.8  Batch: 3  Sleep: 8s


## 1. Helper Utilities

In [7]:
DAILY_QUOTA_EXHAUSTED = False

def _is_rpm_error(msg: str) -> bool:
    low = msg.lower()
    return '429' in msg and ('rate_limit' in low or 'rate limit' in low or 'too many requests' in low)

def _is_hard_quota_error(msg: str) -> bool:
    low = msg.lower()
    return '429' in msg and 'exceeded' in low and 'rate_limit' not in low

def _retry_wait(msg: str, default: int = 35) -> int:
    # Groq returns 'try again in X.Xs' in the error message
    m = re.search(r'try again in ([\d.]+)s', msg)
    if m:
        return int(float(m.group(1))) + 2
    m = re.search(r'retry.after.*?(\d+)', msg.lower())
    if m:
        return int(m.group(1)) + 2
    return default


def call_llm(prompt: str,
             few_shot: Optional[List[Dict]] = None,
             max_retries: int = 3) -> str:
    """
    Groq-backed LLM caller (OpenAI-compatible chat format).
    few_shot: list of {role, parts:[{text:...}]} dicts.
    RPM errors do NOT burn a retry attempt — just sleep and retry same model.
    Returns response text, or '' on failure.
    """
    global active_model_idx, DAILY_QUOTA_EXHAUSTED
    if DAILY_QUOTA_EXHAUSTED:
        print('   🚫 All models exhausted.')
        return ''

    def build_messages(prompt_text):
        messages = []
        if few_shot:
            for turn in few_shot:
                messages.append({
                    'role': turn['role'],
                    'content': turn['parts'][0]['text']
                })
        messages.append({'role': 'user', 'content': prompt_text})
        return messages

    attempt = 0
    while attempt < max_retries:
        attempt += 1
        model = current_model()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=build_messages(prompt),
                temperature=0.2,
                max_tokens=8192,
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            msg = str(e)
            print(f'   ⚠️  [{model}] attempt {attempt}: {msg[:180]}')

            if _is_rpm_error(msg):
                wait = _retry_wait(msg)
                print(f'   ⏱️  RPM limit → sleeping {wait}s...')
                time.sleep(wait)
                attempt -= 1  # don't count RPM waits as a failed attempt
            elif _is_hard_quota_error(msg):
                next_idx = active_model_idx + 1
                if next_idx < len(MODEL_CHAIN):
                    print(f'   🔄 Quota on {model} → switching to {MODEL_CHAIN[next_idx]}')
                    active_model_idx = next_idx
                    attempt = 0
                else:
                    DAILY_QUOTA_EXHAUSTED = True
                    print('   🚫 All models exhausted.')
                    return ''
            elif attempt < max_retries:
                print(f'      retrying in 5s...')
                time.sleep(5)
            else:
                print(f'   ❌ Fatal after {max_retries} attempts.')
    return ''


def parse_json(raw: str) -> Any:
    """Strip markdown code fences then parse JSON."""
    text = re.sub(r'^```[a-z]*\n?', '', raw.strip())
    text = re.sub(r'\n?```$', '', text.strip()).strip()
    return json.loads(text)

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f'   💾 Saved → {path}')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def batched(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

print('✅ Helpers ready.')

✅ Helpers ready.


In [8]:
print('Available Groq models:')
try:
    for m in client.models.list().data:
        print(f'  {m.id}')
except Exception as e:
    print(f'Could not list models: {e}')

print('\nSmoke test...')
test = call_llm('Reply with just the word: OK')
print(f'Response : "{test}"')
print(f'Model    : {current_model()}')

Available Groq models:
  openai/gpt-oss-120b
  whisper-large-v3
  llama-3.3-70b-versatile
  openai/gpt-oss-safeguard-20b
  groq/compound-mini
  llama-3.1-8b-instant
  meta-llama/llama-4-scout-17b-16e-instruct
  moonshotai/kimi-k2-instruct
  meta-llama/llama-prompt-guard-2-86m
  openai/gpt-oss-20b
  moonshotai/kimi-k2-instruct-0905
  canopylabs/orpheus-arabic-saudi
  canopylabs/orpheus-v1-english
  meta-llama/llama-prompt-guard-2-22m
  groq/compound
  whisper-large-v3-turbo
  qwen/qwen3-32b
  allam-2-7b

Smoke test...
Response : "OK"
Model    : llama-3.3-70b-versatile


## 2. Load Ground Truth

In [9]:
ground_truth = load_json(GROUND_TRUTH_PATH)
all_operations: List[Dict] = []
for sec in ground_truth['sections']:
    for op in sec['operations']:
        all_operations.append({
            'sectionId': sec['sectionId'],
            'sectionTitle': sec['sectionTitle'],
            **op
        })

print(f'Ground truth: {len(ground_truth["sections"])} sections, {len(all_operations)} operations')
for s in ground_truth['sections']:
    print(f'  [{s["sectionId"]}] {s["sectionTitle"]} — {len(s["operations"])} ops')

Ground truth: 4 sections, 21 operations
  [1] Authentication — 3 ops
  [2] Plumber Onboarding Wizard (Steps 1–6) — 1 ops
  [3] Plumber Qualification — 7 ops
  [4] Complaint Lifecycle — 10 ops


## 3. LLM 1 — Formal Spec Generator

Replicates the 3-step chain-of-thought from `working.ipynb`:
- **Step 1 — Annotation**: Identify actionable requirement sentences from each operation.
- **Step 2 — Lifting**: Convert NL sentences to structured Lifted-NL temporal/logical statements.
- **Step 3 — Translation**: Convert Lifted-NL to full formal specs (Precondition, API call, Postcondition) in predicate/set-theory notation.

In [10]:
# ── STEP 1: ANNOTATION ────────────────────────────────────────────

ANNOT_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId': {'type': 'string'},
            'operationName': {'type': 'string'},
            'actionable_sentences': {
                'type': 'array', 'items': {'type': 'string'},
                'description': 'Sentences encoding state transitions, guards, or system actions.'
            }
        },
        'required': ['operationId', 'operationName', 'actionable_sentences']
    }
}, indent=2)

FS_ANNOT = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'name': 'Signup',
        'purpose': 'Register a new account.',
        'precondition': 'The username u must not already exist in R.',
        'postcondition': 'R now contains an entry mapping u to hashed password p.'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'actionable_sentences': [
            'The username u must not already exist in R.',
            'R now contains an entry mapping u to hashed password p.'
        ]
    }])}]}
]

def step1_annotate(operations: List[Dict], extra: str = '') -> List[Dict]:
    print('\n🔹 STEP 1: Annotation')
    sys_p = f"""You are an expert in software engineering and formal methods.
Extract ONLY sentences that are actionable system requirements:
  • State transitions  ("state becomes X", "advances to Y", "reverts to Z")
  • Guards/Preconditions ("must be in state X", "must not already exist", "must match")
  • Postconditions ("is removed", "is created", "field is set to", "added to map")
Ignore introductory or descriptive sentences.
{extra}
Output ONLY a raw JSON array (no markdown, no preamble) matching this schema:
{ANNOT_SCHEMA}"""

    results = []
    bs = list(batched(operations, BATCH_SIZE))
    for i, batch in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        print(f'   Batch {i+1}/{len(bs)}: {len(batch)} ops')
        inp = [{'operationId': op['operationId'], 'name': op['name'],
                'purpose': op['purpose'], 'precondition': op['precondition'],
                'postcondition': op['postcondition']} for op in batch]
        raw = call_llm(sys_p + '\nJSON input:\n' + json.dumps(inp, indent=2), few_shot=FS_ANNOT)
        if not raw: continue
        try:
            r = parse_json(raw)
            if not isinstance(r, list): r = [r]
            results.extend(r)
            print(f'   ✅ {sum(len(x.get("actionable_sentences",[])) for x in r)} sentences')
        except Exception as e:
            print(f'   ❌ Parse: {e} | raw[:150]: {raw[:150]}')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)
    print(f'   Total annotated ops: {len(results)}')
    return results

print('✅ Step 1 defined.')

✅ Step 1 defined.


In [11]:
# ── STEP 2: LIFTING ───────────────────────────────────────────────

LIFT_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId': {'type': 'string'},
            'operationName': {'type': 'string'},
            'source_sentence': {'type': 'string'},
            'lifted_nl': {
                'type': 'string',
                'description': "Structured logical statement: 'always','whenever','implies','becomes','dom()',map notation."
            }
        },
        'required': ['operationId', 'operationName', 'source_sentence', 'lifted_nl']
    }
}, indent=2)

LIFT_SYS = f"""You are an expert in formal methods and temporal logic.
Convert each actionable sentence to a Lifted Natural Language (Lifted NL) statement.
Rules:
  • Modal vocabulary: 'always','whenever','implies','must hold that','iff'
  • State changes: 'X becomes Y', 'X is added to map M', 'X is removed from M'
  • Named state vars: T (session token map), Qual (qualification map), Compl (complaint map),
    C/P/M (credential maps), QualState, ComplaintRecord
  • Actors: customer c, plumber p, manager m
  • One logical sentence per item.
Output ONLY a raw JSON array (no markdown, no preamble) matching:
{LIFT_SCHEMA}"""

FS_LIFT = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'The username u must not already exist in R.'
    }, {
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'R now contains an entry mapping u to hashed password p.'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'The username u must not already exist in R.',
        'lifted_nl': 'whenever signup(u,p,role) is called, it must hold that u \u2209 dom(R) where R=roleMap(role)'
    }, {
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'R now contains an entry mapping u to hashed password p.',
        'lifted_nl': 'always after signup(u,p,role) succeeds, R becomes R \u222a {u \u21a6 hash(p)}'
    }])}]}
]

def step2_lift(annotated: List[Dict]) -> List[Dict]:
    print('\n🔹 STEP 2: Lifting')
    flat = [{'operationId': x['operationId'], 'operationName': x['operationName'],
             'source_sentence': s}
            for x in annotated for s in x.get('actionable_sentences', [])]
    print(f'   Sentences to lift: {len(flat)}')
    results = []
    bs = list(batched(flat, BATCH_SIZE))
    for i, batch in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        print(f'   Batch {i+1}/{len(bs)}: {len(batch)} sentences')
        raw = call_llm(LIFT_SYS + '\nSentences:\n' + json.dumps(batch, indent=2), few_shot=FS_LIFT)
        if not raw: continue
        try:
            r = parse_json(raw)
            if not isinstance(r, list): r = [r]
            results.extend(r)
            print(f'   ✅ {len(r)} lifted')
        except Exception as e:
            print(f'   ❌ Parse: {e} | raw[:150]: {raw[:150]}')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)
    print(f'   Total lifted: {len(results)}')
    return results

print('✅ Step 2 defined.')

✅ Step 2 defined.


In [12]:
# ── STEP 3: TRANSLATION ───────────────────────────────────────────

TRANS_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId':   {'type': 'string'},
            'operationName': {'type': 'string'},
            'LABEL': {'type': 'string', 'description': 'ALL_CAPS_SNAKE_CASE e.g. SIGNUP_OK'},
            'lifted_nl': {'type': 'string'},
            'Precondition': {
                'type': 'string',
                'description': 'Boolean predicate. Uses set/predicate logic symbols. No prose.'
            },
            'Function': {
                'type': 'string',
                'description': 'name(param:Type) -> ReturnType [HTTP_CODE]'
            },
            'Postcondition': {
                'type': 'string',
                'description': "New state via primed vars (R',T',Qual',Compl'). List ALL maps."
            }
        },
        'required': ['operationId','operationName','LABEL','lifted_nl',
                     'Precondition','Function','Postcondition']
    }
}, indent=2)

TRANS_SYS = f"""You are an expert Formal Specification Translator (predicate logic + set theory).

GLOBAL STATE:
  C,P,M : Map<username,hashedPassword>  (Customer/Plumber/Manager credentials)
  T     : Map<token,username>           (active session tokens)
  Qual  : Map<plumberId,QualState>
    QualState in {{STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED}}
  Compl : Map<complaintId,ComplaintRecord>
    ComplaintRecord.state in {{RAISED,ASSIGNED,UNDER_EXAM_C,QUOTATION_RAISED,UNDER_EXEC,PAYMENT_PENDING,DONE,ABANDONED}}

RULES:
  1. LABEL        : ALL_CAPS_SNAKE_CASE unique per operation (e.g. LOGIN_OK)
  2. Precondition : pure Boolean predicate, no prose. Use set/predicate logic notation.
  3. Function     : name(param1:Type1, ...) -> ReturnType [HTTP_CODE]
  4. Postcondition: use primed vars, list ALL four maps even if unchanged
     e.g. T'=T union {{tok|->u}} AND C'=C AND Qual'=Qual AND Compl'=Compl
  5. Merge ALL lifted_nl items for the SAME operationId into ONE output record.

Output ONLY a raw JSON array (no markdown, no preamble) matching:
{TRANS_SCHEMA}"""

FS_TRANS = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'lifted_nl': 'whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'LABEL': 'SIGNUP_OK',
        'lifted_nl': 'whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}',
        'Precondition': 'role in {Customer,Plumber,Manager} AND R=roleMap(role) AND u not in dom(R)',
        'Function': 'signup(u:Username, p:Password, role:Role) -> Unit [201 Created]',
        'Postcondition': "R'=R union {u|->hash(p)} AND T'=T AND Qual'=Qual AND Compl'=Compl"
    }])}]}
]

def step3_translate(lifted: List[Dict]) -> List[Dict]:
    print('\n🔹 STEP 3: Translation -> Formal Specs')
    grouped = defaultdict(list)
    for x in lifted: grouped[x['operationId']].append(x)
    op_groups = list(grouped.values())
    print(f'   Operations to formalise: {len(op_groups)}')
    results = []
    bs = list(batched(op_groups, BATCH_SIZE))
    for i, batch_ops in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        items = [x for ops in batch_ops for x in ops]
        print(f'   Batch {i+1}/{len(bs)}: {len(batch_ops)} ops ({len(items)} items)')
        raw = call_llm(TRANS_SYS + '\nLifted-NL:\n' + json.dumps(items, indent=2), few_shot=FS_TRANS)
        if not raw: continue
        try:
            r = parse_json(raw)
            if not isinstance(r, list): r = [r]
            results.extend(r)
            print(f'   ✅ {len(r)} specs')
        except Exception as e:
            print(f'   ❌ Parse: {e} | raw[:150]: {raw[:150]}')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)
    print(f'   Total formal specs: {len(results)}')
    return results

print('✅ Step 3 defined.')

✅ Step 3 defined.


In [13]:
print('='*60)
print('LLM 1 — FORMAL SPEC GENERATOR (initial run)')
print('='*60)

annotated_ops = step1_annotate(all_operations)
lifted_items  = step2_lift(annotated_ops)
formal_specs  = step3_translate(lifted_items)

save_json(formal_specs, FORMAL_SPECS_PATH)

if DAILY_QUOTA_EXHAUSTED:
    print('\n🚫 Halted — quota exhausted on all models.')
else:
    print(f'\n✅ LLM 1 done. {len(formal_specs)} specs -> {FORMAL_SPECS_PATH}')

LLM 1 — FORMAL SPEC GENERATOR (initial run)

🔹 STEP 1: Annotation
   Batch 1/7: 3 ops
   ✅ 6 sentences
   Batch 2/7: 3 ops
   ✅ 7 sentences
   Batch 3/7: 3 ops
   ✅ 6 sentences
   Batch 4/7: 3 ops
   ✅ 6 sentences
   Batch 5/7: 3 ops
   ✅ 7 sentences
   Batch 6/7: 3 ops
   ✅ 6 sentences
   Batch 7/7: 3 ops
   ✅ 6 sentences
   Total annotated ops: 21

🔹 STEP 2: Lifting
   Sentences to lift: 44
   Batch 1/15: 3 sentences
   ✅ 3 lifted
   Batch 2/15: 3 sentences
   ❌ Parse: Invalid \escape: line 18 column 91 (char 729) | raw[:150]: [
  {
    "operationId": "1-B",
    "operationName": "Login",
    "source_sentence": "A new token is issued and recorded in T, mapping token to userna
   Batch 3/15: 3 sentences
   ✅ 3 lifted
   Batch 4/15: 3 sentences
   ✅ 3 lifted
   Batch 5/15: 3 sentences
   ✅ 3 lifted
   Batch 6/15: 3 sentences
   ✅ 3 lifted
   Batch 7/15: 3 sentences
   ✅ 3 lifted
   Batch 8/15: 3 sentences
   ❌ Parse: Invalid \escape: line 12 column 84 (char 482) | raw[:150]: [
  {
    "

## 4. LLM 2 — Summariser

Takes the formal specifications and reconstructs a `draft_summary.json` that mirrors the schema of `summary.json` (sections → operations → precondition / name / postcondition).

In [14]:
SUM_PROMPT = """You are an expert technical writer.
Given formal specs, reconstruct a human-readable summary JSON in EXACTLY this schema:
{
  "sections": [
    {
      "sectionId": <int>,
      "sectionTitle": <string>,
      "operations": [
        {
          "operationId": <string — MUST match input>,
          "name": <string>,
          "purpose": <string — 1-2 plain-English sentences>,
          "stateVariables": [<string>, ...],
          "precondition": <string — plain English, no math>,
          "postcondition": <string — plain English, no math>
        }
      ]
    }
  ]
}
Section grouping:
  1 — Authentication        (1-A, 1-B, 1-C)
  2 — Plumber Onboarding    (2-k)
  3 — Plumber Qualification (3-A..3-C3)
  4 — Complaint Lifecycle   (4-A..4-F)
Include EVERY operationId. Output ONLY raw JSON — no markdown, no preamble.
"""

def run_llm2(specs: List[Dict]) -> Dict:
    print('\n'+'='*60+'\nLLM 2 — SUMMARISER\n'+'='*60)
    if DAILY_QUOTA_EXHAUSTED: return {'sections': []}
    raw = call_llm(SUM_PROMPT + '\nSpecs:\n' + json.dumps(specs, indent=2))
    if not raw: return {'sections': []}
    try:
        d = parse_json(raw)
        n = sum(len(s.get('operations', [])) for s in d.get('sections', []))
        print(f'   ✅ {len(d.get("sections",[]))} sections, {n} ops')
        return d
    except Exception as e:
        print(f'   ❌ Parse: {e}')
        return {'sections': []}

print('✅ LLM 2 defined.')

✅ LLM 2 defined.


## 5. LLM 3 — Judge

Compares `draft_summary.json` to the `ground_truth` summary.json and produces:
- A **similarity score** (0.0 – 1.0)
- Per-section and per-operation breakdown
- Feedback on what is missing or incorrect (used for refinement)

In [15]:
JUDGE_PROMPT = """You are an independent expert judge.
Score each operation in DRAFT vs GROUND TRUTH on these weighted dimensions (0.0-1.0):
  A) name_accuracy       0.15
  B) purpose_accuracy    0.20
  C) precondition_match  0.30
  D) postcondition_match 0.30
  E) state_variables     0.05
operation_score = 0.15A + 0.20B + 0.30C + 0.30D + 0.05E
overall_score   = mean(all operation_scores)

Output EXACTLY this JSON — no markdown, no preamble:
{
  "overall_score": <float 0.0-1.0>,
  "operation_scores": [
    {
      "operationId": <string>,
      "name": <string>,
      "score": <float>,
      "dimension_scores": {
        "name_accuracy": <float>,
        "purpose_accuracy": <float>,
        "precondition_match": <float>,
        "postcondition_match": <float>,
        "state_variables": <float>
      },
      "feedback": <string>
    }
  ],
  "missing_operations": [<operationId>, ...],
  "overall_feedback": <string — top 3 issues to fix>
}
"""

def run_llm3(gt: Dict, draft: Dict) -> Dict:
    print('\n'+'='*60+'\nLLM 3 — JUDGE\n'+'='*60)
    if DAILY_QUOTA_EXHAUSTED:
        return {'overall_score': 0.0, 'overall_feedback': 'Quota exhausted.'}
    prompt = (JUDGE_PROMPT
              + '\n\n--- GROUND TRUTH ---\n' + json.dumps(gt, indent=2)
              + '\n\n--- DRAFT ---\n' + json.dumps(draft, indent=2))
    raw = call_llm(prompt)
    if not raw: return {'overall_score': 0.0, 'overall_feedback': 'Empty response.'}
    try:
        r = parse_json(raw)
        print(f'   📊 Score: {r.get("overall_score", 0):.3f}')
        print(f'   💬 {r.get("overall_feedback", "")[:200]}')
        if r.get('missing_operations'):
            print(f'   ⚠️  Missing: {r["missing_operations"]}')
        return r
    except Exception as e:
        print(f'   ❌ Parse: {e}')
        return {'overall_score': 0.0, 'overall_feedback': f'Parse error: {e}'}

print('✅ LLM 3 defined.')

✅ LLM 3 defined.


## 6. Refinement: LLM 1 Feedback Loop

When the judge score is below threshold, the judge's feedback is fed back into LLM 1
to regenerate the formal specs with targeted corrections. The loop runs until:
- `score >= SIMILARITY_THRESHOLD`, OR
- `MAX_ITERATIONS` is reached.

In [16]:
# def build_refinement_prompt(operations: List[Dict], judge_report: Dict) -> str:
#     """
#     Builds a targeted refinement prompt for LLM 1 using judge feedback.
#     Focuses extra attention on low-scoring and missing operations.
#     """
#     missing = judge_report.get("missing_operations", [])
#     low_scoring = [
#         op for op in judge_report.get("operation_scores", [])
#         if op.get("score", 1.0) < 0.75
#     ]
#     feedback = judge_report.get("overall_feedback", "")

#     low_ids = {op["operationId"] for op in low_scoring}
#     priority_ops = [op for op in operations if op["operationId"] in low_ids or op["operationId"] in missing]

#     feedback_block = "\n".join(
#         f"  - [{op['operationId']}] {op.get('feedback', '')}"
#         for op in low_scoring
#     )

#     refinement_note = f"""
# REFINEMENT INSTRUCTIONS (from judge feedback — iteration):

# Overall feedback: {feedback}

# Operations needing improvement:
# {feedback_block if feedback_block else '  (none specifically flagged)'}

# Missing operations to add: {missing if missing else '(none)'}

# Focus on these operationIds in your formal spec output: {list(low_ids | set(missing))}
# Pay special attention to:
# - Precise precondition predicates (match the ground-truth state variables exactly)
# - Complete postconditions (list ALL state variables, even unchanged ones with prime notation)
# - Correct state names (e.g. UNDER_EXAM, ONBOARD_DONE, PAYMENT_PENDING)
# """
#     return refinement_note


# print("✅ Refinement helper defined.")
def build_refinement(report: Dict) -> str:
    missing  = report.get('missing_operations', [])
    feedback = report.get('overall_feedback', '')
    low_ops  = [op for op in report.get('operation_scores', []) if op.get('score', 1.0) < 0.75]
    lines    = '\n'.join(
        f"  [{op['operationId']}] {op.get('score',0):.2f}: {op.get('feedback','')}"
        for op in low_ops
    )
    focus = sorted(set([op['operationId'] for op in low_ops] + missing))
    return f"""
=== REFINEMENT (from judge) ===
Overall feedback: {feedback}
Low-scoring ops:
{lines or '  (none)'}
Missing: {missing or '(none)'}
Focus on operationIds: {focus}
Fix:
  • Exact QualState names: STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED
  • Exact Compl.state: RAISED,ASSIGNED,UNDER_EXAM_C,QUOTATION_RAISED,UNDER_EXEC,PAYMENT_PENDING,DONE,ABANDONED
  • Postconditions must list ALL four maps (C'/P'/M', T', Qual', Compl') even if unchanged
=== END ===
"""

print('✅ Refinement helper defined.')

✅ Refinement helper defined.


## 7. Main Iterative Loop

In [17]:
print('\n' + '='*60)
print(f'  ITERATIVE PIPELINE  threshold={SIMILARITY_THRESHOLD}  max_iter={MAX_ITERATIONS}')
print('='*60)

iter_log      = []
current_score = 0.0
current_specs = formal_specs

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f'\n--- ITERATION {iteration}/{MAX_ITERATIONS} ---')

    if DAILY_QUOTA_EXHAUSTED:
        print('🚫 Quota exhausted — stopping.')
        break

    draft  = run_llm2(current_specs)
    save_json(draft, DRAFT_SUMMARY_PATH)

    report = run_llm3(ground_truth, draft)
    current_score = report.get('overall_score', 0.0)
    report['iteration'] = iteration
    save_json(report, JUDGE_REPORT_PATH)

    iter_log.append({'iteration': iteration, 'score': current_score,
                     'feedback': report.get('overall_feedback', '')})

    flag = '✅' if current_score >= SIMILARITY_THRESHOLD else '❌'
    print(f'\n  {flag} Score: {current_score:.3f}  (threshold: {SIMILARITY_THRESHOLD})')

    if current_score >= SIMILARITY_THRESHOLD:
        print('  🎉 THRESHOLD REACHED!')
        break

    if iteration == MAX_ITERATIONS:
        print('  ⚠️  Max iterations reached.')
        break

    print(f'  🔁 Refining for iteration {iteration+1}...')
    extra = build_refinement(report)
    ann   = step1_annotate(all_operations, extra=extra)
    lift  = step2_lift(ann)
    spec  = step3_translate(lift)
    current_specs = spec
    save_json(current_specs, f'formal_specifications_iter{iteration+1}.json')
    time.sleep(3)

print('\n' + '='*60 + '\n  DONE\n' + '='*60)
for log in iter_log:
    f = '✅' if log['score'] >= SIMILARITY_THRESHOLD else '❌'
    print(f"  Iter {log['iteration']}: {log['score']:.3f} {f}  {log['feedback'][:90]}")
save_json(iter_log, ITER_LOG_PATH)
print(f'\n  Final score: {current_score:.3f}')
print(f'  Result: {"PASSED ✅" if current_score >= SIMILARITY_THRESHOLD else "NOT REACHED ⚠️"}')


  ITERATIVE PIPELINE  threshold=0.8  max_iter=5

--- ITERATION 1/5 ---

LLM 2 — SUMMARISER
   ✅ 4 sections, 21 ops
   💾 Saved → draft_summary.json

LLM 3 — JUDGE
   📊 Score: 0.615
   💬 The draft is mostly complete, but some operations are missing or incomplete. The purpose, precondition, and postcondition of some operations are partially accurate. State variables are mostly complete
   ⚠️  Missing: ['3-C3']
   💾 Saved → judge_report.json

  ❌ Score: 0.615  (threshold: 0.8)
  🔁 Refining for iteration 2...

🔹 STEP 1: Annotation
   Batch 1/7: 3 ops
   ✅ 6 sentences
   Batch 2/7: 3 ops
   ✅ 7 sentences
   Batch 3/7: 3 ops
   ✅ 6 sentences
   Batch 4/7: 3 ops
   ✅ 6 sentences
   Batch 5/7: 3 ops
   ✅ 7 sentences
   Batch 6/7: 3 ops
   ✅ 6 sentences
   Batch 7/7: 3 ops
   ✅ 6 sentences
   Total annotated ops: 21

🔹 STEP 2: Lifting
   Sentences to lift: 44
   Batch 1/15: 3 sentences
   ✅ 3 lifted
   Batch 2/15: 3 sentences
   ✅ 3 lifted
   Batch 3/15: 3 sentences
   ✅ 3 lifted
   Batch 4/15:

KeyboardInterrupt: 

## 8. Inspect Results

In [18]:
print('\n📋 SAMPLE FORMAL SPECS (first 3)')
for sp in current_specs[:3]:
    print(f"\n[{sp.get('LABEL','?')}] {sp.get('operationName','?')} ({sp.get('operationId','?')})")
    print(f"  Pre : {sp.get('Precondition','—')}")
    print(f"  Fn  : {sp.get('Function','—')}")
    print(f"  Post: {sp.get('Postcondition','—')}")


📋 SAMPLE FORMAL SPECS (first 3)

[SIGNUP_OK] Signup (1-A)
  Pre : u ∉ dom(R)
  Fn  : signup(u:Username, p:Password, role:Role) -> Unit [201 Created]
  Post: C'=C AND P'=P AND M'=M ∪ {u → hash(p)} AND T'=T AND Qual'=Qual AND Compl'=Compl

[LOGIN_OK] Login (1-B)
  Pre : p = hash(inv(R)(u)) ∧ u ∉ dom(T)
  Fn  : login(u:Username, p:Password) -> Token [200 OK]
  Post: C'=C AND P'=P AND M'=M AND T'=T ∪ {token → u} AND Qual'=Qual AND Compl'=Compl

[LOGOUT_OK] Logout (1-C)
  Pre : token ∈ dom(T)
  Fn  : logout(token:Token) -> Unit [200 OK]
  Post: C'=C AND P'=P AND M'=M AND T'=T - {token} AND Qual'=Qual AND Compl'=Compl


In [19]:
draft = load_json(DRAFT_SUMMARY_PATH)
print('\n📋 DRAFT SUMMARY')
for s in draft.get('sections', []):
    ops = s.get('operations', [])
    print(f"  §{s.get('sectionId')}: {s.get('sectionTitle')} — {len(ops)} ops")
    for op in ops:
        print(f"    [{op.get('operationId')}] {op.get('name')}")


📋 DRAFT SUMMARY
  §1: Authentication — 3 ops
    [1-A] Signup
    [1-B] Login
    [1-C] Logout
  §2: Plumber Onboarding — 1 ops
    [2-k] Submit Onboarding Step k
  §3: Plumber Qualification — 6 ops
    [3-A] Initial Qualification Application
    [3-B1] Semi-Qualify (Send to Training)
    [3-B2] Fully Qualify (Activate)
    [3-B3] Disqualify (Reject)
    [3-C1] Suspend Plumber
    [3-C2] Reinstate Plumber
  §4: Complaint Lifecycle — 10 ops
    [4-A] Raise Complaint
    [4-B] Assign Plumber to Complaint
    [4-C1] Plumber Accepts Assignment
    [4-C2] Plumber Rejects Assignment
    [4-D1] Raise Quotation
    [4-D2] Approve Quotation
    [4-D3] Reject Quotation / Abandon
    [4-E1] Mark Task Completed
    [4-E2] Make Payment
    [4-F] Submit Feedback


In [20]:
report = load_json(JUDGE_REPORT_PATH)
print('\n📊 JUDGE REPORT')
print(f"  Score   : {report.get('overall_score')}")
print(f"  Feedback: {report.get('overall_feedback', '')}")
print(f"  Missing : {report.get('missing_operations', [])}")
print('\n  Per-operation:')
for op in report.get('operation_scores', []):
    f = '✅' if op.get('score', 0) >= 0.75 else '⚠️ '
    ds = op.get('dimension_scores', {})
    print(f"  {f} [{op.get('operationId')}] {op.get('name')}: {op.get('score',0):.2f}"
          f" pre={ds.get('precondition_match',0):.2f}"
          f" post={ds.get('postcondition_match',0):.2f}"
          f" — {op.get('feedback','')[:70]}")


📊 JUDGE REPORT
  Score   : 0.73
  Feedback: The draft is generally well-structured, but could benefit from more detailed descriptions of purpose, precondition, and postcondition for many operations. Additionally, state variables could be more thoroughly defined in several cases.
  Missing : []

  Per-operation:
  ✅ [1-A] Signup: 0.81 pre=0.80 post=0.80 — Purpose and precondition could be more detailed.
  ✅ [1-B] Login: 0.83 pre=0.80 post=0.80 — Precondition and postcondition could be more detailed.
  ✅ [1-C] Logout: 0.85 pre=0.90 post=0.90 — State variables could be more detailed.
  ⚠️  [2-k] Submit Onboarding Step k: 0.65 pre=0.60 post=0.60 — Purpose, precondition, postcondition, and state variables could be mor
  ✅ [3-A] Initial Qualification Application: 0.78 pre=0.80 post=0.80 — Purpose and precondition could be more detailed.
  ✅ [3-B1] Semi-Qualify (Send to Training): 0.80 pre=0.80 post=0.80 — Precondition and postcondition could be more detailed.
  ✅ [3-B2] Fully Qualify (Activ